In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/data.csv")   
df.head()

,house_type,locality,city,area,beds,bathrooms,balconies,furnishing,area_rate,rent
0,"2 BHK Flat for Rent in Oberoi Woods, Goregaon ...",Goregaon East,Mumbai,897.0,2,2,0,Semi-Furnished,134.0,120000.0
1,"1 BHK Flat for Rent in Sapphire Lakeside, Powa...",Powai,Mumbai,490.0,1,1,0,Semi-Furnished,82.0,40000.0
2,1 BHK House for Rent in Mundhwa Pune,Mundhwa,Pune,550.0,1,1,0,Unfurnished,22.0,12000.0
3,"2 BHK Flat for Rent in Hingna, Nagpur",Hingna,Nagpur,1000.0,2,2,0,Unfurnished,8.0,8000.0
4,1 BHK Flat for Rent in Unique Star Harsh Vihar...,Mira Road,Mumbai,595.0,1,1,0,Unfurnished,25.0,15000.0


In [2]:
df["rent"].sort_values(ascending=False).head(15)

3073    2700000.0
1276    1600000.0
7575    1500000.0
3066    1500000.0
5534    1500000.0
3600    1450000.0
1363    1350000.0
4774    1300000.0
6822    1210000.0
6834    1200000.0
7238    1000000.0
1253    1000000.0
3900    1000000.0
6066    1000000.0
249     1000000.0
Name: rent, dtype: float64

In [3]:
print("rent over 5 lakh:", (df["rent"] > 500000).sum())
print("rent over 3 lakh:", (df["rent"] > 300000).sum())
print("rent over 2 lakh:", (df["rent"] > 200000).sum())
print("total rows:", df.shape[0])

rent over 5 lakh: 54
rent over 3 lakh: 153
rent over 2 lakh: 297
total rows: 7691


In [4]:
before = df.shape[0]
clean = df[df["rent"] <= 300000].copy()
after = clean.shape[0]
print("Before:", before, " After:", after, " Cut:", before - after)

Before: 7691  After: 7538  Cut: 153


In [5]:
furnished = clean[clean["furnishing"] == "Furnished"]["rent"]
unfurnished = clean[clean["furnishing"] == "Unfurnished"]["rent"]

print("Furnished   — count:", len(furnished),   "avg rent: ₹", round(furnished.mean()))
print("Unfurnished — count:", len(unfurnished), "avg rent: ₹", round(unfurnished.mean()))

Furnished   — count: 1548 avg rent: ₹ 55430
Unfurnished — count: 2636 avg rent: ₹ 35732


In [6]:
from scipy import stats

t_stat, p_value = stats.ttest_ind(furnished, unfurnished, equal_var=False)

print("t-statistic:", round(t_stat, 2))
print("p-value:", p_value)

t-statistic: 12.56
p-value: 2.886783176948241e-35


In [7]:
# FINDING: furnished flats rent ~₹19,700 more than unfurnished. Welch t-test p=3e-35, highly significant. (gap is real, but cause not proven - could be size/location)

In [8]:
clean.groupby("beds")["rent"].mean().round()

beds
1      17833.0
2      32322.0
3      68139.0
4     123519.0
5     122500.0
6      62500.0
7     136667.0
8     115000.0
10    117667.0
Name: rent, dtype: float64

In [9]:
clean["beds"].value_counts().sort_index()

beds
1     1946
2     3043
3     2005
4      460
5       66
6        6
7        3
8        3
10       6
Name: count, dtype: int64

In [10]:
from scipy import stats

# keep only trustworthy bedroom counts
bhk_data = clean[clean["beds"].isin([1, 2, 3, 4])]

# make a separate rent list for each bedroom count
g1 = bhk_data[bhk_data["beds"] == 1]["rent"]
g2 = bhk_data[bhk_data["beds"] == 2]["rent"]
g3 = bhk_data[bhk_data["beds"] == 3]["rent"]
g4 = bhk_data[bhk_data["beds"] == 4]["rent"]

# ANOVA: are these 4 groups all the same, or is at least one different?
f_stat, p_value = stats.f_oneway(g1, g2, g3, g4)

print("F-statistic:", round(f_stat, 2))
print("p-value:", p_value)

F-statistic: 1341.3
p-value: 0.0


In [11]:
# FINDING 2: rent differs significantly by bedroom count (1-4 BHK). ANOVA F=1341, p~0. Strong link, but size likely confounds - not proven causal.

In [12]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Post-hoc on 1-4 BHK only (same subset as the ANOVA above).
bhk_tukey = pairwise_tukeyhsd(endog=bhk_data["rent"], groups=bhk_data["beds"], alpha=0.05)
print(bhk_tukey)
print(f"\nSignificant BHK pairs: {bhk_tukey.reject.sum()} of {len(bhk_tukey.reject)} comparisons")

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
group1 group2   meandiff  p-adj    lower       upper    reject
--------------------------------------------------------------
     1      2  14489.4774   0.0  11651.9309  17327.0239   True
     1      3  50306.3185   0.0  47195.4365  53417.2005   True
     1      4 105685.5371   0.0 100617.3095 110753.7646   True
     2      3  35816.8411   0.0  33004.8746  38628.8075   True
     2      4  91196.0597   0.0  86305.6099  96086.5094   True
     3      4  55379.2186   0.0  50325.2681  60433.1691   True
--------------------------------------------------------------

Significant BHK pairs: 6 of 6 comparisons


In [13]:
#city-wise rent sorting
clean.groupby("city")["rent"].mean().round().sort_values(ascending=False)

city
Mumbai       73376.0
Bangalore    50397.0
New Delhi    35468.0
Pune         30816.0
Nagpur       18016.0
Name: rent, dtype: float64

In [14]:
from scipy import stats

mumbai    = clean[clean["city"] == "Mumbai"]["rent"]
bangalore = clean[clean["city"] == "Bangalore"]["rent"]
delhi     = clean[clean["city"] == "New Delhi"]["rent"]
pune      = clean[clean["city"] == "Pune"]["rent"]
nagpur    = clean[clean["city"] == "Nagpur"]["rent"]

f_stat, p_value = stats.f_oneway(mumbai, bangalore, delhi, pune, nagpur)

print("F-statistic:", round(f_stat, 2))
print("p-value:", p_value)

F-statistic: 285.47
p-value: 6.735329659558885e-229


In [15]:
# FINDING 3: rent differs significantly across 5 cities. ANOVA F=285, p~e-229. Mumbai highest (~73k), Nagpur lowest (~18k), ~4x spread.

In [16]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Post-hoc: WHICH city pairs differ? (ANOVA only says "at least one does")
city_tukey = pairwise_tukeyhsd(endog=clean["rent"], groups=clean["city"], alpha=0.05)
print(city_tukey)
print(f"\nSignificant city pairs: {city_tukey.reject.sum()} of {len(city_tukey.reject)} comparisons")

         Multiple Comparison of Means - Tukey HSD, FWER=0.05         
  group1    group2    meandiff  p-adj     lower       upper    reject
---------------------------------------------------------------------
Bangalore    Mumbai  22979.6064    0.0   18758.684  27200.5287   True
Bangalore    Nagpur -32381.0257    0.0 -38194.7898 -26567.2616   True
Bangalore New Delhi -14928.7127    0.0 -19044.3355 -10813.0899   True
Bangalore      Pune -19580.8856    0.0 -23705.1428 -15456.6283   True
   Mumbai    Nagpur -55360.6321    0.0 -61239.2745 -49481.9896   True
   Mumbai New Delhi  -37908.319    0.0 -42115.0916 -33701.5465   True
   Mumbai      Pune -42560.4919    0.0 -46775.7122 -38345.2717   True
   Nagpur New Delhi   17452.313    0.0  11648.8139  23255.8121   True
   Nagpur      Pune  12800.1401    0.0   6990.5146  18609.7657   True
New Delhi      Pune  -4652.1729 0.0172  -8761.9475   -542.3983   True
---------------------------------------------------------------------

Significant city pa

### Post-hoc (Tukey HSD)

ANOVA only says *at least one* group differs. Tukey HSD pins down *which* pairs, controlling the family-wise error rate across all comparisons.

- **City — all 10 of 10 pairs differ significantly** (p-adj < 0.05). Even the closest pair, New Delhi vs Pune (~₹4,650 gap), is significant (p-adj ≈ 0.017). Mumbai sits above every other city; Nagpur below every other.
- **BHK — all 6 of 6 pairs (1–4 BHK) differ significantly** (p-adj ≈ 0). Each bedroom step is a real jump: 1→2 ≈ ₹14.5k, 2→3 ≈ ₹35.8k, 3→4 ≈ ₹55.4k.

Both ANOVA findings hold at the pairwise level — every city and every adjacent BHK step is statistically distinguishable.

In [17]:
clean.to_csv("data/rentals_clean.csv", index=False)
print("Saved:", clean.shape)

Saved: (7538, 10)


## Phase 4 — Statistical Findings

1. **Furnishing matters.** Furnished flats rent ~₹19,700 more than unfurnished.
   Welch's t-test: t=12.56, p≈3e-35. Significant.

2. **Bedrooms drive rent.** Rent rises clearly across 1–4 BHK.
   One-way ANOVA: F=1341, p≈0. Significant. (Restricted to 1–4 BHK; 5+ too sparse.)

3. **Cities differ sharply.** Mumbai (~₹73k) highest, Nagpur (~₹18k) lowest — ~4x spread.
   One-way ANOVA: F=285, p≈6e-229. Significant.

*Note: all three show statistical association, not proven causation. Bedroom/city
effects likely partly reflect flat size and location.*